# Extended Covariance Verification

This notebook verifies the extended (9x9) coordinate covariance support in `adam_core`.

Orbits fitted with non-gravitational parameters carry a single covariance over the
fixed basis `(coordinates, A1, A2, A3)` in `Orbits.coordinates.covariance`. It focuses
on three questions:

- Is the full 9x9 covariance preserved through coordinate conversions (with the
  non-grav block invariant and cross-covariances rotating with the orbital Jacobian)?
- Does the leading 6x6 block always agree with the ordinary coordinate covariance?
- Can variant creation and grouped collapse round-trip the full 9x9, keeping
  non-gravitational parameters aligned with the sampled orbital state?

The examples below use a synthetic case and real SBDB / NEOCC fixtures already
checked into the repository.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from adam_core.coordinates import (
    CartesianCoordinates,
    CoordinateCovariances,
    KeplerianCoordinates,
    Origin,
    transform_coordinates,
)
from adam_core.orbits import Orbits, VariantOrbits
from adam_core.orbits.non_gravitational_parameters import NonGravitationalParameters
from adam_core.orbits.query.neocc import _parse_oef
from adam_core.orbits.query.sbdb import _orbits_from_sbdb_payloads
from adam_core.time import Timestamp


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "adam_core").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing src/adam_core")


ROOT = find_repo_root(Path.cwd().resolve())
TESTDATA = ROOT / "src" / "adam_core" / "orbits" / "query" / "tests" / "testdata"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


## Synthetic Case: 9x9 Keplerian Covariance Round-Trip

We create a Keplerian orbit whose covariance spans the six orbital elements plus
`A1, A2, A3` (with non-zero cross-covariances), convert it to Cartesian and back,
and verify:

- the full 9x9 matrix survives the round-trip
- the non-grav block is exactly invariant under the transforms
- the leading 6x6 block equals the coordinate covariance transformed on its own


In [ ]:
keplerian_covariance = np.diag(
    [1e-10, 2e-10, 3e-8, 4e-8, 5e-8, 6e-8, 9e-26, 4e-26, 1e-26]
).reshape(1, 9, 9)
keplerian_covariance[0, 1, 6] = keplerian_covariance[0, 6, 1] = 2e-18
keplerian_covariance[0, 5, 7] = keplerian_covariance[0, 7, 5] = -1e-18
keplerian_covariance[0, 3, 8] = keplerian_covariance[0, 8, 3] = 5e-19

keplerian = KeplerianCoordinates.from_kwargs(
    a=[1.2],
    e=[0.15],
    i=[7.0],
    raan=[30.0],
    ap=[45.0],
    M=[12.0],
    covariance=CoordinateCovariances.from_matrix(keplerian_covariance),
    time=Timestamp.from_mjd([60000.0], scale="tdb"),
    origin=Origin.from_kwargs(code=["SUN"]),
    frame="ecliptic",
)

cartesian = transform_coordinates(keplerian, CartesianCoordinates)
back = transform_coordinates(cartesian, KeplerianCoordinates)

cartesian_full = cartesian.covariance.to_full_matrix()[0]
back_full = back.covariance.to_full_matrix()[0]

display(
    pd.DataFrame(
        {
            "metric": [
                "non-grav block max |Cartesian - input|",
                "round trip max relative diagonal error",
                "leading 6x6 == to_matrix()",
            ],
            "value": [
                f"{np.max(np.abs(cartesian_full[6:, 6:] - keplerian_covariance[0, 6:, 6:])):.3e}",
                f"{np.max(np.abs(np.diag(back_full) - np.diag(keplerian_covariance[0])) / np.diag(keplerian_covariance[0])):.3e}",
                bool(
                    np.array_equal(
                        cartesian_full[:6, :6], cartesian.covariance.to_matrix()[0]
                    )
                ),
            ],
        }
    )
)


## Synthetic Case: Joint Sampling and Collapse

We sample variants from the 9x9 covariance (19 sigma points for the 9-dimensional
state) and collapse them back, verifying the recovered covariance — including the
non-grav variances and cross-covariances — matches the input.


In [ ]:
orbits = Orbits.from_kwargs(
    orbit_id=["synthetic"],
    object_id=["synthetic"],
    non_gravitational_parameters=NonGravitationalParameters.from_kwargs(
        source=["manual"], A1=[1e-13], A2=[-2e-13], A3=[4e-13]
    ),
    coordinates=cartesian,
)

variants = VariantOrbits.create(orbits, method="sigma-point")
collapsed = variants.collapse(orbits)

recovered = collapsed.coordinates.covariance.to_full_matrix()[0]
original = cartesian.covariance.to_full_matrix()[0]

display(
    pd.DataFrame(
        {
            "metric": [
                "number of variants (2 * 9 + 1)",
                "A2 spread across variants",
                "max relative diagonal error after collapse",
            ],
            "value": [
                len(variants),
                f"{np.ptp(variants.non_gravitational_parameters.A2.to_numpy(zero_copy_only=False)):.3e}",
                f"{np.max(np.abs(np.diag(recovered) - np.diag(original)) / np.diag(original)):.3e}",
            ],
        }
    )
)


## Real Case: SBDB `99942`

Apophis's SBDB solution estimates `A1` and `A2`. Ingestion builds the 9x9 covariance
in the cometary basis and the ordinary `to_cartesian()` transform carries it to the
Cartesian orbit. `A3` was not estimated, so its row/column is zero (held fixed).


In [ ]:
payload = json.loads((TESTDATA / "sbdb" / "99942_phys.json").read_text())
orbit_99942 = _orbits_from_sbdb_payloads(["99942"], [payload])

full = orbit_99942.coordinates.covariance.to_full_matrix()[0]
display(
    pd.DataFrame(
        {
            "metric": ["sigma(A1) [au/d^2]", "sigma(A2) [au/d^2]", "A3 row all zero"],
            "value": [
                f"{np.sqrt(full[6, 6]):.4e}",
                f"{np.sqrt(full[7, 7]):.4e}",
                bool(np.all(full[8, :] == 0.0)),
            ],
        }
    )
)


## Real Case: NEOCC `99942`

The NEOCC OEF solution estimates `A2` (with a non-estimated `AMRAT` that is dropped
at ingestion). The Keplerian-basis covariance is scaled to canonical units and
extended to the fixed 9x9 layout.


In [ ]:
from unittest.mock import MagicMock, patch

from adam_core.orbits.query.neocc import query_neocc

response = MagicMock()
response.status_code = 200
response.text = (TESTDATA / "neocc" / "99942.ke1").read_text()

with patch("requests.get", return_value=response):
    neocc_99942 = query_neocc(["99942"], orbit_type="ke", orbit_epoch="present-day")

full = neocc_99942.coordinates.covariance.to_full_matrix()[0]
display(
    pd.DataFrame(
        {
            "metric": [
                "A2 [au/d^2]",
                "sigma(A2) [au/d^2]",
                "A1/A3 rows all zero",
            ],
            "value": [
                f"{neocc_99942.non_gravitational_parameters.A2[0].as_py():.4e}",
                f"{np.sqrt(full[7, 7]):.4e}",
                bool(np.all(full[6, :] == 0.0) and np.all(full[8, :] == 0.0)),
            ],
        }
    )
)
